# CSE 144 Final Project — Transfer Learning (100 classes)

Workflow (same spirit as HW2):

1. **Data** — transforms, `ImageFolder` on `data/train/`, train/val split, DataLoaders
2. **Model** — frozen ResNet-18 backbone → new MLP layers → 100-class head; then optional `layer4` fine-tune
3. **Training** — `train_one_epoch` / `evaluate`, checkpointing, plots
4. **Submission** — predict test IDs from `data/sample_submission.csv` → `submission.csv`

In [4]:
# If you are running in a fresh environment, uncomment:
# %pip -q install -r requirements.txt

import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

print("torch:", torch.__version__)
device = "cpu"  # Switch to "cuda" after debugging if you have a GPU.
print("device:", device)

DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")
CKPT_PATH = "./checkpoints/best_model.pt"
SUBMISSION_OUT = "./submission.csv"

IMG_SIZE = 224
NUM_CLASSES = 100
BATCH_SIZE = 16
NUM_WORKERS = 0
VAL_FRACTION = 0.1
EPOCHS_HEAD = 12
EPOCHS_LAYER4 = 10
LR_HEAD = 3e-3
LR_LAYER4 = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 4
TARGET_VAL_ACC = 0.70  # aim for this before Kaggle submit
MIN_VAL_ACC_FOR_SUBMIT = 0.70
HEAD_HIDDEN = (256, 128)  # new trainable layers after frozen backbone
HEAD_DROPOUT = (0.2, 0.1)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def set_seed(seed: int = 42):
    """Make results as reproducible as possible across runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)


set_seed(42)
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

torch: 2.12.0+cpu
device: cpu


c:\Users\maria\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [5]:
# Data pipeline (HW2-style): transforms, ImageFolder, train/val split, DataLoaders

train_tf = transforms.Compose([
    # Keep this simple initially so the head can learn with the frozen backbone.
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ApplyTransform(Dataset):
    """Apply a transform to images from a Subset of ImageFolder (PIL in, tensor out)."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


# ImageFolder: folder name "0".."99" -> label 0..99 (required for Kaggle)
full_train = datasets.ImageFolder(TRAIN_DIR)
assert len(full_train.classes) == NUM_CLASSES, (
    f"Expected {NUM_CLASSES} classes, got {len(full_train.classes)}"
)

# IMPORTANT: ImageFolder sorts class folders alphabetically ("0","1","10",...),
# but the assignment requires numeric label mapping: folder "0" -> label 0, ..., "99" -> 99.
# Remap targets to use the folder name as the integer label.
for i, (path, _old_target) in enumerate(full_train.samples):
    folder = os.path.basename(os.path.dirname(path))
    full_train.samples[i] = (path, int(folder))
full_train.targets = [t for _, t in full_train.samples]
full_train.classes = [str(i) for i in range(NUM_CLASSES)]
full_train.class_to_idx = {str(i): i for i in range(NUM_CLASSES)}

n_val = int(len(full_train) * VAL_FRACTION)
n_train = len(full_train) - n_val
split_gen = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(full_train, [n_train, n_val], generator=split_gen)

train_set = ApplyTransform(train_subset, train_tf)
val_set = ApplyTransform(val_subset, val_tf)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

print("classes (first 5):", full_train.classes[:5])
print("train/val:", len(train_set), len(val_set))
print("train dir exists:", os.path.isdir(TRAIN_DIR))

classes (first 5): ['0', '1', '10', '11', '12']
train/val: 972 107
train dir exists: True


## Model

In [6]:
# Run the **Setup** cell above first (imports + hyperparameters).
from torchvision import models
import torch
import torch.nn as nn

# Fallbacks so this cell works even if Setup wasn't run yet.
# (Still recommended: run Setup -> Data -> Model -> Training.)
if "device" not in globals():
    device = "cpu"
if "NUM_CLASSES" not in globals():
    NUM_CLASSES = 100
if "HEAD_HIDDEN" not in globals():
    HEAD_HIDDEN = (256, 128)
if "HEAD_DROPOUT" not in globals():
    HEAD_DROPOUT = (0.5, 0.3)

# Frozen pretrained backbone + NEW MLP layers + classifier head
# ResNet-18: conv1..layer4 (frozen) -> avgpool -> fc replaced by trainable head

BACKBONE_PARTS = ("conv1", "bn1", "layer1", "layer2", "layer3", "layer4")


def build_classifier_head(in_features, num_classes, hidden_dims, dropouts):
    """New layers after the frozen feature extractor, then the class head."""
    layers = []
    prev = in_features
    for h, drop in zip(hidden_dims, dropouts):
        layers.extend([
            nn.Linear(prev, h),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
        ])
        prev = h
    layers.append(nn.Linear(prev, num_classes))
    return nn.Sequential(*layers)


def set_requires_grad(module, requires_grad: bool):
    for p in module.parameters():
        p.requires_grad = requires_grad


def freeze_pretrained_backbone(m):
    """Freeze ImageNet weights; only the new head (model.fc) trains."""
    for name in BACKBONE_PARTS:
        set_requires_grad(getattr(m, name), False)
    set_requires_grad(m.fc, True)


def unfreeze_layer4_and_head(m):
    """Unfreeze layer3+layer4 + head (keep only earliest layers frozen)."""
    for name in ("layer1", "layer2"):
        set_requires_grad(getattr(m, name), False)
    set_requires_grad(m.layer3, True)
    set_requires_grad(m.layer4, True)
    set_requires_grad(m.fc, True)


def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable


def print_trainable_summary(m):
    print("Trainable modules:")
    for name, child in m.named_children():
        n = sum(p.numel() for p in child.parameters() if p.requires_grad)
        if n > 0:
            print(f"  {name}: {n:,} params")


weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)
in_features = model.fc.in_features
model.fc = build_classifier_head(
    in_features, NUM_CLASSES, HEAD_HIDDEN, HEAD_DROPOUT
)
model = model.to(device)

print("Classifier head (new layers + output):")
print(model.fc)
total, trainable = count_params(model)
print(f"Total params: {total:,} | trainable after freeze: ", end="")
freeze_pretrained_backbone(model)
_, trainable = count_params(model)
print(f"{trainable:,}")
print_trainable_summary(model)

Classifier head (new layers + output):
Sequential(
  (0): Linear(in_features=512, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=128, out_features=100, bias=True)
)
Total params: 11,353,636 | trainable after freeze: 177,124
Trainable modules:
  fc: 177,124 params


## Training

In [7]:
# Training (HW2-style) + early stopping on val accuracy

criterion = nn.CrossEntropyLoss()
optimizer = None
epochs_without_improvement = 0


def train_one_epoch(model, loader):
    """Train for one epoch and return (avg_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model and return (avg_loss, accuracy)."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def run_epochs(model, epochs, lr, phase_name, early_stop=True):
    global optimizer, best_val_acc, best_epoch, epochs_without_improvement
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    print(f"\n--- {phase_name}: up to {epochs} epochs, lr={lr} ---")
    print_trainable_summary(model)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader)
        val_loss, val_acc = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
            f"val loss: {val_loss:.4f} acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = len(history["val_acc"])
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": best_epoch,
                    "val_acc": best_val_acc,
                    "classes": full_train.classes,
                    "head_hidden": HEAD_HIDDEN,
                    "head_dropout": HEAD_DROPOUT,
                },
                CKPT_PATH,
            )
            print(f"  -> saved checkpoint (val acc {best_val_acc:.4f})")
        else:
            epochs_without_improvement += 1

        if best_val_acc >= TARGET_VAL_ACC:
            print(f"  -> reached target val acc {TARGET_VAL_ACC:.0%}; stopping {phase_name}")
            break
        if early_stop and epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  -> no val improvement for {EARLY_STOP_PATIENCE} epochs; stopping")
            break


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = -1
epochs_without_improvement = 0

# Phase 1: backbone frozen (conv1..layer4), train only new head
freeze_pretrained_backbone(model)
run_epochs(model, EPOCHS_HEAD, LR_HEAD, "Phase 1: train new head (backbone frozen)")

# Phase 2: unfreeze ONLY layer4 + head (not the full network)
epochs_without_improvement = 0
unfreeze_layer4_and_head(model)
run_epochs(model, EPOCHS_LAYER4, LR_LAYER4, "Phase 2: fine-tune layer4 + head")

print("\nBest val acc:", f"{best_val_acc:.4f}", "at epoch", best_epoch)
print("Saved to:", CKPT_PATH)
if best_val_acc >= TARGET_VAL_ACC:
    print(f"Ready for Kaggle: val acc >= {TARGET_VAL_ACC:.0%}")
else:
    print(
        f"Not at {TARGET_VAL_ACC:.0%} yet — keep tuning or train longer before submitting."
    )


--- Phase 1: train new head (backbone frozen): up to 12 epochs, lr=0.001 ---
Trainable modules:
  fc: 177,124 params


Epoch 1/12 | train loss: 4.5747 acc: 0.0329 | val loss: 4.3751 acc: 0.0654
  -> saved checkpoint (val acc 0.0654)


Epoch 2/12 | train loss: 4.2333 acc: 0.0607 | val loss: 3.8991 acc: 0.0841
  -> saved checkpoint (val acc 0.0841)


Epoch 3/12 | train loss: 3.9043 acc: 0.0844 | val loss: 3.5750 acc: 0.1028
  -> saved checkpoint (val acc 0.1028)


Epoch 4/12 | train loss: 3.6872 acc: 0.0957 | val loss: 3.4852 acc: 0.1215
  -> saved checkpoint (val acc 0.1215)


...

Epoch 8/12 | train loss: 3.2932 acc: 0.1821 | val loss: 3.0670 acc: 0.2336


Epoch 9/12 | train loss: 3.2141 acc: 0.2212 | val loss: 2.9872 acc: 0.2991
  -> saved checkpoint (val acc 0.2991)


Epoch 10/12 | train loss: 3.0891 acc: 0.2654 | val loss: 2.9592 acc: 0.2804


Epoch 11/12 | train loss: 3.0443 acc: 0.2747 | val loss: 2.8803 acc: 0.3364
  -> saved checkpoint (val acc 0.3364)


KeyboardInterrupt: 

In [ ]:
# Plot training curves (HW2 Q3.4)

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Submission

In [ ]:
# Kaggle submission: predict labels for IDs in sample_submission.csv


class TestImageDataset(Dataset):
    def __init__(self, test_dir, image_ids, transform=None):
        self.test_dir = test_dir
        self.image_ids = list(image_ids)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        fname = self.image_ids[idx]
        path = os.path.join(self.test_dir, fname)
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, fname


submission_df = pd.read_csv(SAMPLE_SUBMISSION)
test_ids = submission_df["ID"].tolist()
print("Submission rows:", len(test_ids))

test_set = TestImageDataset(TEST_DIR, test_ids, transform=val_tf)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

if not os.path.isfile(CKPT_PATH):
    raise FileNotFoundError(f"No checkpoint at {CKPT_PATH}. Run training first.")

checkpoint = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
ckpt_val = checkpoint.get("val_acc", 0.0)
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']}")
print(f"Checkpoint val acc: {ckpt_val:.4f} ({ckpt_val:.1%})")

if ckpt_val < MIN_VAL_ACC_FOR_SUBMIT:
    raise RuntimeError(
        f"Val acc {ckpt_val:.1%} is below {MIN_VAL_ACC_FOR_SUBMIT:.0%}. "
        "Do not submit to Kaggle yet — re-run training after tuning. "
        "To force write submission.csv anyway, lower MIN_VAL_ACC_FOR_SUBMIT in the setup cell."
    )
print(f"Val acc >= {MIN_VAL_ACC_FOR_SUBMIT:.0%} — OK to upload submission.csv to Kaggle.")

model.eval()
all_preds = []
all_ids = []

with torch.no_grad():
    for images, fnames in tqdm(test_loader, desc="predict"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_ids.extend(fnames)

out_df = pd.DataFrame({"ID": all_ids, "Label": all_preds})
assert len(out_df) == len(submission_df)
assert set(out_df["Label"]) <= set(range(NUM_CLASSES))

out_df.to_csv(SUBMISSION_OUT, index=False)
print("Wrote:", SUBMISSION_OUT)
print(out_df.head())